In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

**Create Simulator**

In [3]:
simulator = AerSimulator()

**Generate Bitstrings for Alice and Bob**

In [4]:
def generate_bits_quantum(n_bits): # HELPER FUNCTION TO GENERATE THE BITS so it can be any length..

  qc = QuantumCircuit(1, 1)
  qc.h(0)
  qc.measure(0, 0)

  result = simulator.run(qc, shots=n_bits, memory=True).result() # run n_bits times with shots
  bitstring = ''.join(result.get_memory()) # combine together

  return bitstring

alice_basis=generate_bits_quantum(16)
alice_state=generate_bits_quantum(16)
bob_basis=generate_bits_quantum(16)

print("Randomly generating Alice and Bob...\n")
print(f"alice_basis: {alice_basis}")
print(f"alice_state: {alice_state}")
print(f"bob_basis: {bob_basis}")

Randomly generating Alice and Bob...

alice_basis: 1101101010001001
alice_state: 1011011100001111
bob_basis: 0101111001111101


**Function for Alice and Bob to Send and Receive**

In [5]:
"""
LEGEND for BASIS
0 -> standard basis
1 -> diagonal basis

"""

# For Alice
def sender_function(state, basis):
  bitlength = len(state)

  circuit = QuantumCircuit(bitlength)

  for i in range(bitlength):
    # qubit default is ket 0
    if state[i] == '1': # if bit 1, use pauli-x gate to change to ket 1
      circuit.x(i)

    # after that, check the basis of the bit
    if basis[i] == '1': # if bit 1, use hadamard gate to change into ket - or +
      circuit.h(i)

  return circuit

# For Bob
def receiver_function(message, measurement_basis):
  bitlength = len(measurement_basis)

  for i in range(bitlength):

    if measurement_basis[i] == '1':
      message.h(i)

  message.measure_all()

  # Run the measured circuit
  compiled_circuit = transpile(message, simulator)

  result = simulator.run(
        compiled_circuit,
        shots=1,
        memory=True
    ).result()

  return result.get_memory()[0][::-1]

# For Eve (Attacker)
def attacker_function(message, length):
  eve_basis = generate_bits_quantum(length)

  eve_result = receiver_function(alice_encoded_circuit, eve_basis)

  eve_resent_circuit = sender_function(eve_result, eve_basis)

  return eve_resent_circuit, eve_basis, eve_result

# Simulate Send and Receive
alice_encoded_circuit = sender_function(alice_state, alice_basis)

# Attack in the middle!
eve_resent_circuit, eve_basis, eve_result = attacker_function(
    alice_encoded_circuit,
    16
)

bob_received_circuit = receiver_function(eve_resent_circuit, bob_basis)


**Sifting Function** : Generates both sender and receiver key

In [6]:
def sifting_function(sender_basis, sender_state, receiver_basis, receiver_result):
  sender_key = ""
  receiver_key = ""

  for i in range(len(sender_basis)):
    if sender_basis[i] == receiver_basis[i]:
      sender_key += sender_state[i]
      receiver_key += receiver_result[i]

  return sender_key, receiver_key

sifted_key, receiver_key = sifting_function(alice_basis, alice_state, bob_basis, bob_received_circuit)

**Check Quantum Bit Error Rate**

In [7]:
def calculate_qber(sender_key, receiver_key):
  errors = 0

  for i in range(len(sender_key)):
    if sender_key[i] != receiver_key[i]:
      errors += 1

  if len(sender_key) == 0:
    qber = 0
  else:
    qber = errors / len(sender_key)

  return qber, errors

qber, errors = calculate_qber(sifted_key, receiver_key)
print(f"Quantum Bit Error Rate: {qber}")
print(f"Number of errors: {errors}")

Quantum Bit Error Rate: 0.2222222222222222
Number of errors: 2


Final Results

In [8]:
# Final Conclusion Block

print("\n==============================")
print("BB84 Protocol with Attacker Summary")
print("==============================")

print("\n Bitstring and Basis generation with Quantum Randomness")
print("----------------------------------")

print("\nAlice basis: ", alice_basis)
print("Alice state: ", alice_state)
print("Bob basis:   ", bob_basis)

print("Sifted key:  ", sifted_key)
print("Receiver key:", receiver_key)
print("Sifted key length:", len(sifted_key))

print("\n QBER Checking")
print("----------------")
print("Number of errors:", errors)
print("QBER:", qber)
print("QBER percentage:", qber * 100, "%")

if (qber > 0.11): # Following standard of 11% threshold
  print("Attack detected!")



BB84 Protocol with Attacker Summary

 Bitstring and Basis generation with Quantum Randomness
----------------------------------

Alice basis:  1101101010001001
Alice state:  1011011100001111
Bob basis:    0101111001111101
Sifted key:   011011111
Receiver key: 011001011
Sifted key length: 9

 QBER Checking
----------------
Number of errors: 2
QBER: 0.2222222222222222
QBER percentage: 22.22222222222222 %
Attack detected!
